# Stage C 03f — c15 deep-adaptive checkpoint analysis

Analysis only: this notebook reads the completed c15 deep-adaptive checkpoint, produces a held-out memory trace/PCA and a controlled delayed-association probe, and records them in the study ledger. It does not train, resume, or overwrite c15. Run it on a GPU runtime before deciding whether to start the fresh 5M scale gate.

In [ ]:
# USER CONFIGURATION
REPO_URL='https://github.com/Gonza10V/SeqTrainer.git'
GIT_REF='TO_BE_PINNED_AFTER_COMMIT'
DRIVE_ROOT='/content/drive/MyDrive/SeqTrainerStageC'
SOURCE_RUN_NAME='c15_paper_deep_compact_1m_paper_exact'
TRACE_MAX_STREAMS=8; TRACE_MAX_SEGMENTS=128; BEHAVIOR_PAIRS=32


In [ ]:
from pathlib import Path
from google.colab import drive
import json, subprocess, sys
mount=Path('/content/drive')
if not (mount/'MyDrive').is_dir(): drive.mount(str(mount), timeout_ms=120000)
repo=Path('/content/SeqTrainer')
if not repo.exists(): subprocess.run(['git','clone',REPO_URL,str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'fetch','origin'],check=True)
subprocess.run(['git','-C',str(repo),'checkout',GIT_REF],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',f'{repo}[torch,bacteria-titan]'],check=True)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Use a Colab GPU runtime for this analysis.')
selection=json.loads((Path(DRIVE_ROOT)/'runs/c1_tokenizers_cpu/tokenizer_selection.json').read_text())
dataset=Path(DRIVE_ROOT)/'stage_c_dataset/ordered_streams'/selection['selected_tokenizer']
PROTOCOL=repo/'studies/stage_c_ecoli_escherichia_paper_deep_memory_v2/protocol.json'
STUDY_ROOT=Path(DRIVE_ROOT)/'study/stage_c_ecoli_escherichia_paper_deep_memory_v2'
source_run=Path(DRIVE_ROOT)/'runs'/SOURCE_RUN_NAME/'deep_adaptive'
checkpoint=source_run/'latest.pt'
if not checkpoint.is_file(): raise FileNotFoundError(f'Missing c15 checkpoint: {checkpoint}')
if not (source_run/'run_manifest.json').is_file(): raise FileNotFoundError('Missing c15 run_manifest.json')
analysis_root=source_run/'analysis_c15_v1'
subprocess.run(['seqtrainer-titans-stage-c-study','initialize','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT)],check=True)
print('Reading preserved checkpoint:',checkpoint)


In [ ]:
def run_logged(run_dir,label,command):
    try:
        subprocess.run(['seqtrainer-titans-stage-c-colab-run','--run-dir',str(run_dir),'--label',label,'--repo',str(repo),'--',*command],check=True)
    except subprocess.CalledProcessError:
        for path in (run_dir/'FAILED.txt',run_dir/'logs'/f'{label}.log'):
            if path.exists(): print(path.read_text(errors='replace')[-16000:])
        raise
def record_once(marker_name, run_id, artifact):
    marker=STUDY_ROOT/'record_markers'/marker_name
    if marker.exists():
        print('Ledger record already exists:',marker)
        return
    subprocess.run(['seqtrainer-titans-stage-c-study','record','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT),'--run-id',run_id,'--evidence-tier','exploratory','--artifact',str(artifact)],check=True)
    marker.parent.mkdir(parents=True,exist_ok=True)
    marker.write_text(json.dumps({'run_id':run_id,'artifact':str(artifact)},indent=2)+'\n')
trace_dir=analysis_root/'memory_trace_val'
if not (trace_dir/'memory_trace.json').is_file():
    run_logged(trace_dir,'memory_trace_c15',['seqtrainer-titans-stage-c-memory-trace','--dataset-dir',str(dataset),'--checkpoint',str(checkpoint),'--output-dir',str(trace_dir),'--split','val','--memory-mode','adaptive','--max-streams',str(TRACE_MAX_STREAMS),'--max-segments',str(TRACE_MAX_SEGMENTS),'--device','cuda','--protocol',str(PROTOCOL),'--run-id','deep_memory_trace_analysis'])
record_once('c15_deep_memory_trace_analysis.json','deep_memory_trace_analysis',trace_dir)
behavior=analysis_root/'controlled_memory_behavior.json'
if not behavior.is_file():
    run_logged(analysis_root,'controlled_memory_behavior_c15',['seqtrainer-titans-stage-c-memory-behavior','--checkpoint',str(checkpoint),'--output',str(behavior),'--pairs',str(BEHAVIOR_PAIRS),'--device','cuda','--protocol',str(PROTOCOL),'--run-id','deep_controlled_behavior'])
record_once('c15_deep_controlled_behavior.json','deep_controlled_behavior',behavior)
trace=json.loads((trace_dir/'memory_trace.json').read_text())
probe=json.loads(behavior.read_text())
summary={key:trace[key] for key in ('segments','streams','mean_bits_per_base','mean_memory_update_norm','mean_surprise_norm','pca_explained_variance','pc1_gc_correlation','pc1_stream_position_correlation')}
print('Held-out trace summary:')
print(json.dumps(summary,indent=2,sort_keys=True))
print('Controlled association behavior:')
print(json.dumps({'all_blocks_improve_immediately':probe['all_blocks_improve_immediately'],'blocks_with_positive_delayed_margin':probe['blocks_with_positive_delayed_margin'],'blocks':probe['blocks']},indent=2,sort_keys=True))
print('PCA visualization:',trace_dir/'memory_pca.svg')
print('This is exploratory architecture/learning evidence, not a biological or adaptive-memory-superiority claim.')
